# Lesson 3 — Enabling Self-Service Analytics & AI for Business Users

**Enbridge Supply Chain Management — Databricks Enablement Workshop**

---

### What you'll learn in this lesson

| Section | Topic | Duration |
|---------|-------|---------|
| **A** | Dashboards & Visualizations | ~30 min |
| **B** | Designing Genie Q&A Experiences | ~30 min |
| **C** | MLflow for Model Lifecycle Management | ~30 min |
| **D** | Agentic AI & Generative AI with Mosaic AI | ~30 min |
| **E** | Databricks One for Self-Service Delivery | ~15 min |
| **F** | End-User Access, Security & Governance | ~15 min |

### Prerequisites
* Lesson 2 completed (Gold tables created in `enbridge_scm` schema)
* Tables available: `gold_regional_supplier_coverage`, `gold_agreement_summary`, `gold_contract_analytics`, `silver_preferred_suppliers`, `silver_agreements`

In [0]:
# ── Configuration (same as Lesson 2) ──────────────────────────────────────
dbutils.widgets.text("catalog", "serverless_stable_qr9if1_catalog", "1) Catalog")
dbutils.widgets.text("schema", "enbridge_scm", "2) Schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA  = dbutils.widgets.get("schema").strip()

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print(f"Working in: {CATALOG}.{SCHEMA}")

# Verify Gold and Silver tables exist from Lesson 2
print("Verifying tables from Lesson 2...\n")
original_tables = ['gold_regional_supplier_coverage', 'gold_agreement_summary', 'gold_contract_analytics', 'silver_preferred_suppliers', 'silver_agreements']
enriched_tables = ['gold_supplier_spend_summary', 'gold_category_coverage', 'gold_supplier_compliance', 'silver_supplier_enriched', 'silver_contract_mapping', 'silver_po_spend']

print("Original tables:")
for tbl in original_tables:
    count = spark.table(tbl).count()
    print(f"  {tbl}: {count:,} rows")

print("\nEnriched tables (new datasets):")
for tbl in enriched_tables:
    try:
        count = spark.table(tbl).count()
        print(f"  {tbl}: {count:,} rows")
    except Exception as e:
        print(f"  {tbl}: NOT FOUND - run Lesson 2 Section A.6+ first!")

---
# Section A — Dashboards & Visualizations

> **Goal:** Build clear, interactive dashboards on Gold-level data and publish them
> so business users can explore key metrics independently.

### Best Practices for SCM Dashboards
* **Use Gold tables only** — pre-aggregated, governed, performant
* **SQL Warehouse** as compute — fastest for dashboard queries
* **Parameterize** — let users filter by BU, region, time period
* **Keep it simple** — 5-7 key visualizations per dashboard page
* **Publish with access controls** — share read-only with business users

In [0]:
%sql
-- ============================================================
-- DASHBOARD QUERY 1: Supplier Coverage by Business Unit
-- ============================================================
-- This query powers a bar chart showing how many preferred suppliers
-- each business unit has, split by category managed vs regional sourcing.

SELECT 
  business_unit_name,
  supplier_type,
  SUM(unique_suppliers) AS total_unique_suppliers
FROM gold_regional_supplier_coverage
GROUP BY business_unit_name, supplier_type
ORDER BY total_unique_suppliers DESC;

In [0]:
%sql
-- ============================================================
-- DASHBOARD QUERY 2: Regional Supplier Distribution
-- ============================================================
-- Powers a regional heatmap or treemap visualization

SELECT 
  region,
  bu_code,
  business_unit_name,
  SUM(unique_suppliers) AS suppliers,
  SUM(total_assignments) AS total_assignments
FROM gold_regional_supplier_coverage
GROUP BY region, bu_code, business_unit_name
ORDER BY suppliers DESC;

In [0]:
%sql
-- ============================================================
-- DASHBOARD QUERY 3: Agreement Status Overview
-- ============================================================
-- Powers a donut chart or KPI cards showing agreement health

SELECT 
  document_type,
  agreement_status,
  total_agreements,
  unique_suppliers,
  total_contracts
FROM gold_agreement_summary
ORDER BY total_agreements DESC;

In [0]:
%sql
-- ============================================================
-- DASHBOARD QUERY 4: Contract Analytics by Business Unit
-- ============================================================
-- Powers a stacked bar chart of contract counts by BU and status

SELECT 
  business_unit,
  contract_status,
  SUM(contract_count) AS total_contracts,
  SUM(unique_suppliers) AS unique_suppliers
FROM gold_contract_analytics
GROUP BY business_unit, contract_status
ORDER BY total_contracts DESC;

## A.1b — NEW: Dashboard Queries for Enriched Datasets

These queries use the new Gold tables built from the enriched PSL Dashboard datasets.
They power additional dashboard panels for spend analysis, category coverage, and compliance.

In [0]:
%sql
-- ============================================================
-- NEW DASHBOARD QUERY 5: Top Suppliers by Total Spend
-- ============================================================
-- Powers a horizontal bar chart of top 15 suppliers by total PO spend.

-- Excludes cancelled / withdrawn / rejected POs so spend reflects real commitments.
SELECT 
  supplier_name,
  SUM(total_spend)     AS total_spend,
  SUM(num_line_items)  AS num_line_items,
  SUM(num_documents)   AS num_documents
FROM gold_supplier_spend_summary
WHERE document_status NOT IN ('CANCELED', 'WITHDRAWN', 'REJECTED')
GROUP BY supplier_name
ORDER BY total_spend DESC
LIMIT 15;

In [0]:
%sql
-- ============================================================
-- NEW DASHBOARD QUERY 6: L3 Category Coverage Heatmap
-- ============================================================
-- Powers a heatmap showing how many suppliers cover each L3 category
-- by business unit. Identifies coverage gaps.

SELECT 
  business_unit_name,
  l3_category,
  SUM(unique_suppliers) AS total_suppliers,
  SUM(linked_contracts) AS total_contracts
FROM gold_category_coverage
GROUP BY business_unit_name, l3_category
ORDER BY total_suppliers DESC
LIMIT 30;

In [0]:
%sql
-- ============================================================
-- NEW DASHBOARD QUERY 7: Supplier Compliance Status
-- ============================================================
-- Powers a donut chart showing compliance status distribution.

SELECT 
  compliance_status,
  count(*) AS num_responses,
  round(count(*) * 100.0 / sum(count(*)) OVER (), 1) AS pct
FROM gold_supplier_compliance
GROUP BY compliance_status
ORDER BY num_responses DESC;

## A.2 — Creating a Dashboard from These Queries

### Step-by-step: Build in Databricks
1. Navigate to **SQL Editor** → Run each query above
2. Click **"+ New Dashboard"** from the query results
3. Add visualizations:
   * **Bar chart** → Supplier Coverage by BU (Query 1)
   * **Treemap** → Regional Distribution (Query 2)
   * **Donut chart** → Agreement Status (Query 3)
   * **Stacked bar** → Contract Analytics (Query 4)
4. Add **filter widgets** for `business_unit_name` and `region`
5. **Publish** and share with the team

### Migrating from Power BI
The "Preferred Supplier List" Power BI report maps directly to our Gold tables:

| Power BI Report Element | Databricks Dashboard Widget | Source Table |
|------------------------|----------------------------|-------------|
| Supplier by Region | Treemap by region | `gold_regional_supplier_coverage` |
| Agreement Overview | KPI cards + donut | `gold_agreement_summary` |
| Contract Summary | Stacked bar by BU | `gold_contract_analytics` |
| Supplier Search | Data table with filters | `silver_preferred_suppliers` |

> **💡 Demo:** The instructor will create a live dashboard during the session.

---
# Section B — Designing and Building Genie Q&A Experiences

> **Goal:** Configure Genie on curated datasets so business users can ask questions
> in plain language and get consistent, accurate answers.

### What is Genie?
Genie is Databricks' **natural language interface** to your data. Users type questions like:
* "How many preferred suppliers does Liquids Pipelines have in the PRAIRIE region?"
* "Show me all OPEN agreements for supplier 100343"
* "Which business unit has the most contracts?"

...and Genie translates to SQL, runs it, and returns results.

### Keys to a Great Genie Experience
1. **Use Gold tables** — pre-aggregated, well-named columns
2. **Add column comments** — Genie uses them to understand semantics
3. **Define semantic context** — teach Genie your business terminology
4. **Set guardrails** — restrict which columns/tables are accessible
5. **Test with real questions** — validate answers against known results

## B.1 — Preparing Tables for Genie

Genie works best when tables have:
* ✅ Descriptive **column comments** (we added these in Lesson 2)
* ✅ Clean **column names** (snake_case, no spaces)
* ✅ **Business-friendly values** (full names, not codes)
* ✅ A manageable **table size** (Gold tables, not raw Bronze)

Let's verify our Gold tables are Genie-ready:

In [0]:
%sql
-- ============================================================
-- GENIE PREP: Verify Gold tables have proper column descriptions
-- ============================================================
-- Genie uses column comments to understand what each field means.
-- Let's add any missing descriptions.

-- Add comments to Gold tables for Genie context
ALTER TABLE gold_regional_supplier_coverage ALTER COLUMN bu_code 
  COMMENT 'Business unit code: LP (Liquids Pipelines), GTM (Gas Transmission), GPT (Power Operations)';
ALTER TABLE gold_regional_supplier_coverage ALTER COLUMN business_unit_name 
  COMMENT 'Full business unit name';
ALTER TABLE gold_regional_supplier_coverage ALTER COLUMN unique_suppliers 
  COMMENT 'Count of distinct preferred suppliers in this BU/region/type combination';
ALTER TABLE gold_regional_supplier_coverage ALTER COLUMN supplier_type 
  COMMENT 'How the supplier is managed: Category Managed or Regional Sourcing';

ALTER TABLE gold_agreement_summary ALTER COLUMN document_type 
  COMMENT 'Agreement type: Blanket Purchase Agreement (BPA) or Contract Purchase Agreement (CPA)';
ALTER TABLE gold_agreement_summary ALTER COLUMN total_agreements 
  COMMENT 'Total number of purchase agreements';

ALTER TABLE gold_contract_analytics ALTER COLUMN business_unit 
  COMMENT 'Enbridge business unit: LP, GTM, GDS, GPT';
ALTER TABLE gold_contract_analytics ALTER COLUMN contract_count 
  COMMENT 'Total number of contracts';

-- Verify
DESCRIBE TABLE gold_regional_supplier_coverage;

## B.2 — Creating a Genie Space

### Step-by-step: Build a Genie Space

1. Navigate to **Genie** in the left sidebar
2. Click **"+ New"** to create a Genie Space
3. **Name it:** "Enbridge SCM — Preferred Supplier Analytics"
4. **Add tables:**
   * `gold_regional_supplier_coverage`
   * `gold_agreement_summary`
   * `gold_contract_analytics`
   * `silver_preferred_suppliers` (for detailed lookups)
5. **Add instructions** (semantic context):

```
You are a supply chain analytics assistant for Enbridge.

Business units:
- LP = Liquids Pipelines
- GTM = Gas Transmission  
- GDS = Enbridge Gas
- GPT = Power Operations

Supplier types:
- "Category Managed" = suppliers managed centrally by category
- "Regional Sourcing" = suppliers managed by regional teams

Agreement types:
- BPA = Blanket Purchase Agreement (multi-use)
- CPA = Contract Purchase Agreement (specific contract)

When asked about "preferred suppliers", use silver_preferred_suppliers.
When asked about "agreements" or "contracts", use gold_agreement_summary.
When asked about "coverage" or "by region", use gold_regional_supplier_coverage.
```

6. **Test with sample questions:**
   * "How many preferred suppliers does GTM have?"
   * "Show me all regions for Liquids Pipelines"
   * "Which agreement type has the most OPEN agreements?"
   * "Top 10 suppliers by number of agreements"

> **💡 Demo:** The instructor will create a live Genie Space and test questions.

---
# Section C — MLflow for Model Lifecycle Management

> **Goal:** Track experiments, evaluate model performance, and register models for deployment.

### MLflow in a Nutshell
* **Experiment Tracking** — log parameters, metrics, and artifacts for every model run
* **Model Registry** — version, stage, and deploy models
* **Evaluation** — automated model quality assessment
* **Deployment** — serve models as REST endpoints

### SCM Use Case: Supplier Risk Classification
We'll build a simple model that predicts **supplier risk** based on:
* Number of agreements (more = established = lower risk)
* Agreement status distribution (many CLOSED = higher risk)
* Regional coverage (broader = more diversified = lower risk)

In [0]:
from pyspark.sql import functions as F
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import numpy as np

# ============================================================
# FEATURE ENGINEERING: Build supplier features from Gold tables
# ============================================================

# Feature 1: Supplier coverage breadth
supplier_coverage = spark.sql("""
  SELECT 
    supplier_number,
    supplier_name,
    COUNT(DISTINCT region) AS num_regions,
    COUNT(DISTINCT bu_code) AS num_business_units,
    MAX(supplier_type) AS primary_supplier_type
  FROM silver_preferred_suppliers
  GROUP BY supplier_number, supplier_name
""").toPandas()

# Feature 2: Agreement activity
supplier_agreements = spark.sql("""
  SELECT 
    supplier_number,
    COUNT(DISTINCT agreement_number) AS num_agreements,
    SUM(CASE WHEN agreement_status = 'OPEN' THEN 1 ELSE 0 END) AS open_agreements,
    SUM(CASE WHEN agreement_status = 'CLOSED' THEN 1 ELSE 0 END) AS closed_agreements,
    SUM(CASE WHEN agreement_status = 'ON HOLD' THEN 1 ELSE 0 END) AS onhold_agreements
  FROM silver_agreements
  GROUP BY supplier_number
""").toPandas()

# Merge features
features = supplier_coverage.merge(supplier_agreements, on='supplier_number', how='left')
# Fill only numeric columns -- filling text columns with 0 creates mixed types Spark cannot convert
num_cols = features.select_dtypes(include='number').columns
features[num_cols] = features[num_cols].fillna(0)
features['supplier_name'] = features['supplier_name'].fillna('Unknown').astype(str)
features['primary_supplier_type'] = features['primary_supplier_type'].fillna('Unknown').astype(str)

# Create risk label (for demo: suppliers with few regions and few agreements = higher risk)
features['risk_score'] = (
    (features['num_regions'] < 2).astype(int) +
    (features['num_agreements'] < 3).astype(int) +
    (features['closed_agreements'] > features['open_agreements']).astype(int)
)
features['risk_category'] = features['risk_score'].apply(
    lambda x: 'High' if x >= 2 else ('Medium' if x == 1 else 'Low')
)

print(f"Feature dataset: {len(features)} suppliers")
print(f"\nRisk distribution:")
print(features['risk_category'].value_counts())
display(spark.createDataFrame(features).limit(10))

In [0]:
# ============================================================
# NEW: Enriched Features from PSL Dashboard Datasets
# ============================================================
# Adds spend, contract mapping, and compliance features to the model.

# Feature 3: Total spend from PO line items (50K rows)
spend_features = spark.sql("""
  SELECT 
    vendor_name AS supplier_name,
    CAST(SUM(total_ordered_amount) AS DOUBLE) AS total_spend,
    COUNT(*) AS num_po_lines,
    COUNT(DISTINCT document_number) AS num_po_documents
  FROM silver_po_spend
  WHERE vendor_name IS NOT NULL
  GROUP BY vendor_name
""").toPandas()

# Feature 4: L3 category breadth from contract mapping
category_features = spark.sql("""
  SELECT 
    supplier_name,
    COUNT(DISTINCT l3_category) AS num_l3_categories,
    COUNT(DISTINCT region) AS num_contract_regions,
    COUNT(DISTINCT contract_number) AS num_linked_contracts
  FROM silver_contract_mapping
  GROUP BY supplier_name
""").toPandas()

# Feature 5: Compliance status
compliance_features = spark.sql("""
  SELECT 
    supplier_name,
    SUM(CASE WHEN compliance_status = 'Compliant' THEN 1 ELSE 0 END) AS compliant_responses,
    SUM(CASE WHEN compliance_status = 'Non-Compliant' THEN 1 ELSE 0 END) AS noncompliant_responses,
    SUM(CASE WHEN compliance_status = 'Pending' THEN 1 ELSE 0 END) AS pending_responses
  FROM gold_supplier_compliance
  GROUP BY supplier_name
""").toPandas()

# Merge enriched features into the existing feature set
features = features.merge(spend_features, on='supplier_name', how='left').fillna(0)
features = features.merge(category_features, on='supplier_name', how='left').fillna(0)
features = features.merge(compliance_features, on='supplier_name', how='left').fillna(0)

# Update risk score to include spend and compliance factors
features['risk_score'] = (
    (features['num_regions'] < 2).astype(int) +
    (features['num_agreements'] < 3).astype(int) +
    (features['closed_agreements'] > features['open_agreements']).astype(int) +
    (features['total_spend'] < 10000).astype(int) +
    (features['noncompliant_responses'] > 0).astype(int)
)
features['risk_category'] = features['risk_score'].apply(
    lambda x: 'High' if x >= 3 else ('Medium' if x >= 1 else 'Low')
)

print(f"Enriched feature dataset: {len(features)} suppliers")
print(f"\nUpdated risk distribution:")
print(features['risk_category'].value_counts())
print(f"\nNew feature columns: total_spend, num_po_lines, num_l3_categories, num_linked_contracts, compliant_responses, noncompliant_responses")

In [0]:
# ============================================================
# MLFLOW: Train a Supplier Risk Classifier with Experiment Tracking
# ============================================================

# Prepare training data
feature_cols = ['num_regions', 'num_business_units', 'num_agreements',
                'open_agreements', 'closed_agreements', 'onhold_agreements',
                'total_spend', 'num_po_lines', 'num_l3_categories',
                'num_linked_contracts', 'compliant_responses', 'noncompliant_responses']
X = features[feature_cols]
y = features['risk_category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Set up MLflow experiment
experiment_name = f"/Users/{spark.conf.get('spark.databricks.workspaceUrl', 'default')}/enbridge_scm_supplier_risk"
try:
    mlflow.set_experiment(experiment_name)
except:
    mlflow.set_experiment("/enbridge_scm_supplier_risk")

# Train with MLflow tracking
with mlflow.start_run(run_name="rf_supplier_risk_enriched_v2") as run:
    # Log parameters
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("features", feature_cols)
    mlflow.log_param("num_suppliers", len(features))
    
    # Train model
    model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
    model.fit(X_train, y_train)
    
    # Evaluate
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # Log metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("train_size", len(X_train))
    mlflow.log_metric("test_size", len(X_test))
    
    # Log model
    mlflow.sklearn.log_model(model, "supplier_risk_model")
    
    # Log feature importance
    importance = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n✅ Model trained and logged to MLflow")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Accuracy: {accuracy:.2%}")
    print(f"\nFeature Importance:")
    print(importance.to_string(index=False))
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred))

## C.3 — MLflow Key Concepts Recap

### What We Just Did
1. **Feature Engineering** — Built supplier features from Gold/Silver tables
2. **Experiment Tracking** — Logged parameters, metrics, and the model to MLflow
3. **Evaluation** — Accuracy score and classification report

### Next Steps in a Real Project
| Step | Action | MLflow feature |
|------|--------|---------------|
| **Register** | Promote model to Model Registry | `mlflow.register_model()` |
| **Stage** | Move through Dev → Staging → Production | Model aliases ("Champion", "Challenger") |
| **Deploy** | Serve as REST API | Model Serving endpoint |
| **Monitor** | Track drift and performance | Inference tables + monitoring |

> **💡 Advanced exercise:** Register the model and create a serving endpoint
> to provide real-time supplier risk scores via API.

---
# Section D — Agentic AI & Generative AI with Mosaic AI

> **Goal:** Introduction to building AI-powered agents and generative AI applications
> in Databricks using Mosaic AI and enterprise data.

### Mosaic AI Building Blocks

| Capability | What it does | SCM use case |
|-----------|-------------|-------------|
| **AI Functions (SQL)** | Call LLMs directly from SQL | Classify suppliers, summarize contracts |
| **Vector Search** | Semantic search over enterprise data | "Find suppliers similar to X" |
| **AI Gateway** | Governed access to LLMs | Rate limiting, logging, fallback |
| **Agent Framework** | Build multi-step AI agents | Contract Q&A bot, spend analysis assistant |
| **MLflow for GenAI** | Track prompts, models, evaluations | Log and compare LLM configurations |

### Enbridge SCM AI Use Cases
* **Supplier Analytics Agent:** "Analyze supplier X's performance across regions"
* **Contract Intelligence:** Upload a contract PDF, extract key terms, compare to database
* **Spend Analysis Assistant:** "What categories have the highest spend growth?"
* **Reporting Assistant:** Natural language → dashboard generation

In [0]:
%sql
-- ============================================================
-- AI FUNCTION: Classify supplier risk using AI from SQL
-- ============================================================
-- Databricks AI functions let you call LLMs directly from SQL.
-- This is the simplest way to add AI to your data pipelines.

-- Example: Generate a risk summary for suppliers with many agreements
SELECT 
  supplier_name,
  num_agreements,
  open_agreements,
  closed_agreements,
  ai_gen(
    CONCAT(
      'Provide a one-sentence risk assessment for supplier "', 
      supplier_name, 
      '" who has ', 
      CAST(num_agreements AS STRING), 
      ' agreements (', 
      CAST(open_agreements AS STRING), ' open, ',
      CAST(closed_agreements AS STRING), ' closed).'
    )
  ) AS ai_risk_assessment
FROM (
  SELECT 
    s.supplier_name,
    COUNT(DISTINCT a.agreement_number) AS num_agreements,
    SUM(CASE WHEN a.agreement_status = 'OPEN' THEN 1 ELSE 0 END) AS open_agreements,
    SUM(CASE WHEN a.agreement_status = 'CLOSED' THEN 1 ELSE 0 END) AS closed_agreements
  FROM silver_preferred_suppliers s
  LEFT JOIN silver_agreements a ON s.supplier_number = a.supplier_number
  GROUP BY s.supplier_name
  HAVING num_agreements > 0
)
ORDER BY num_agreements DESC
LIMIT 5;

In [0]:
%sql
-- ============================================================
-- AI FUNCTION: Extract structured data from contract descriptions
-- ============================================================
-- Use ai_extract() to pull structured fields from free-text descriptions.

SELECT 
  `Contract Number`,
  `Contract Header Name`,
  `Contract Header Description`,
  ai_extract(
    `Contract Header Description`,
    '{
      "service_category": {"type": "string", "description": "Main service or product category"},
      "duration_months": {"type": "integer", "description": "Contract duration in months if mentioned"},
      "has_safety_clause": {"type": "boolean", "description": "True if safety or compliance is mentioned"}
    }'
  ) AS extracted_fields
FROM bronze_contracts
WHERE `Contract Header Description` IS NOT NULL
  AND LENGTH(`Contract Header Description`) > 20
LIMIT 5;

## D.2 — Building AI Agents with Mosaic AI

### Agent Architecture Overview
```
User Question → Agent (LLM) → Tools (SQL, Vector Search, Dashboards) → Response
```

### Building a Supplier Q&A Agent (Conceptual)

1. **Define tools** the agent can use:
   * `query_supplier_data` — SQL on `silver_preferred_suppliers`
   * `query_agreements` — SQL on `silver_agreements`
   * `search_contracts` — Vector search on `bronze_contracts` descriptions

2. **System prompt** with Enbridge context:
```
You are a supply chain assistant for Enbridge.
You help analyze preferred suppliers, agreements, and contracts.
Business units: LP (Liquids Pipelines), GTM (Gas Transmission), GPT (Power Operations).
Always cite the data source and row counts in your responses.
```

3. **Deploy** as an AI Gateway endpoint or Databricks App

### Governance for GenAI
* **AI Gateway** — rate limiting, token quotas, request logging
* **MLflow Tracing** — log every agent interaction for audit
* **Vector Search** — RAG grounded in enterprise data (no hallucinations)
* **Unity Catalog** — agents inherit table-level permissions

> **💡 Demo:** The instructor will show a live AI agent answering SCM questions.
> **💡 Advanced exercise:** Use the Databricks Agent Framework to build a simple
> supplier lookup agent with one SQL tool.

---
# Section E — Genie One for Self-Service Delivery

> **Goal:** Use Databricks One to provide a unified, user-friendly portal for analytics.

### What is Genie One?
Genie One is a simplified interface for business users — no Databricks experience needed.
It exposes:
* **Dashboards** — interactive visualizations
* **Genie Spaces** — natural language Q&A
* **Catalog Browser** — discoverable, governed datasets

### How Business Users Access It
1. Admin shares a Genie One URL with the team
2. Users log in with SSO (no Databricks workspace access needed)
3. They see only what they have permission to:
   * Approved dashboards
   * Assigned Genie Spaces
   * Catalog tables they can read

### Enbridge SCM Delivery Plan
| Asset | Where it lives | Who gets access |
|------|--------------|---------------|
| Preferred Supplier Coverage dashboard | Databricks dashboards → Genie One | All SCM team |
| Agreement Status dashboard | Databricks dashboards → Genie One | SCM analysts + managers |
| SCM Genie Space | Genie → Genie One | All SCM team (read-only) |
| Contract Analytics dashboard | Databricks dashboards → Genie One | SCM managers + finance |

> **💡 Demo:** The instructor will show the Genie One interface and how
> business users interact with shared assets.

---
# Section F — End-User Access, Security & Governance

> **Goal:** Best practices for publishing and governing self-service analytics assets.

### Access Model for Enbridge SCM

```
Unity Catalog (3-level namespace)
└── serverless_stable_qr9if1_catalog
    └── enbridge_scm
        ├── bronze_*    → Data Engineers only (WRITE + READ)
        ├── silver_*    → Data Engineers (WRITE) + Analysts (READ)
        ├── gold_*     → Everyone in SCM team (READ)
        └── views       → Analysts (READ)
```

### Key Governance Features

| Feature | Purpose | Enbridge use case |
|---------|---------|------------------|
| **GRANT statements** | Control who can read/write | Bronze = engineers only; Gold = all SCM |
| **Row-level security** | Filter rows by user | Show only suppliers for user's BU/region |
| **Column masking** | Hide sensitive columns | Mask supplier contact info for business users |
| **Tags** | Classification & search | domain=supply_chain, layer=gold |
| **Lineage** | Track data flow | Audit trail from Bronze → Gold |
| **Audit logs** | Monitor access | system.access.audit_log table |

In [0]:
%sql
-- ============================================================
-- GOVERNANCE: Grant access to Gold tables for business users
-- ============================================================
-- In production, you would create groups and grant access:

/*
-- Create a group for SCM business users
CREATE GROUP IF NOT EXISTS scm_business_users;

-- Grant read-only access to Gold tables
GRANT SELECT ON TABLE gold_regional_supplier_coverage TO scm_business_users;
GRANT SELECT ON TABLE gold_agreement_summary TO scm_business_users;
GRANT SELECT ON TABLE gold_contract_analytics TO scm_business_users;

-- Grant access to Silver (read-only) for analysts
CREATE GROUP IF NOT EXISTS scm_analysts;
GRANT SELECT ON TABLE silver_preferred_suppliers TO scm_analysts;
GRANT SELECT ON TABLE silver_agreements TO scm_analysts;

-- Bronze: engineers only (no grants to business users)
-- They can only access via Gold tables and dashboards
*/

SELECT 'Access control examples shown above (uncomment in production)' AS note;

In [0]:
%sql
-- ============================================================
-- GOVERNANCE: Row-Level Security (RLS) Example
-- ============================================================
-- Row filters ensure users only see data for their business unit.
-- This is applied via a function + GRANT.

/*
-- Create a function that filters by the current user's BU
CREATE OR REPLACE FUNCTION enbridge_scm.filter_by_user_bu(region_value STRING)
RETURNS BOOLEAN
RETURN 
  -- Allow full access for data engineers
  is_member('scm_data_engineers')
  -- For business users, only show their assigned BU
  OR (
    is_member('scm_business_users') 
    AND region_value IN (
      SELECT region FROM enbridge_scm.user_bu_mapping 
      WHERE user_email = current_user()
    )
  );

-- Apply the row filter to the Gold table
ALTER TABLE gold_regional_supplier_coverage
SET ROW FILTER enbridge_scm.filter_by_user_bu(region);
*/

SELECT 'Row-level security example shown above (uncomment in production)' AS note;

---
# 🎯 Hands-On Exercise (Lesson 3)

### Exercise 1: Build a Dashboard Query (10 min)
Write a SQL query on `gold_contract_analytics` that shows:
* Total contracts per business unit
* Filtered to only `Active` status
* Ordered by contract count descending

### Exercise 2: Prepare a Table for Genie (10 min)
Choose one Gold table and:
* Add column comments for at least 5 columns
* Write 3 sample questions a business user might ask
* Verify the table is Genie-ready using DESCRIBE TABLE

### Exercise 3: AI Enhancement (10 min)
Use `ai_gen()` to generate a one-paragraph supplier summary for the top 5 suppliers
by agreement count. Include their agreement status distribution in the prompt.

### 🌟 Stretch: RLS Setup (5 min)
Write a row filter function that would restrict `gold_regional_supplier_coverage`
to only show rows where `bu_code = 'LP'` for a specific user group.

---
# Lesson 3 — Summary

### What We Covered

| Section | Topic | Key takeaway |
|---------|-------|-------------|
| **A** | Dashboards | Build on Gold tables, publish via Databricks One |
| **A (enriched)** | Spend, Category Coverage, Compliance dashboards | New Gold tables from PSL Dashboard data |
| **B** | Genie | Add column comments + instructions -> natural language Q&A |
| **C** | MLflow | Track experiments, log metrics, register models |
| **C (enriched)** | Enriched ML features | Added spend, category, compliance features to risk model |
| **D** | Agentic AI | AI functions in SQL, agent framework, governed LLM access |
| **E** | Databricks One | Unified portal for business users -- no Databricks experience needed |
| **F** | Security | GRANT, RLS, column masking, tags, audit logs |

### Architecture: From Source to Self-Service
```
Oracle (ENBOAC) → Bronze → Silver → Gold → Dashboards + Genie + ML + AI
                                                      ↓
                                               Genie One
                                                      ↓
                                            Business Users (SCM Team)
```

### Key Enbridge Assets Delivered
* 13 Bronze tables (5 original + 8 enriched)
* 5 Silver tables (2 original + 3 enriched)
* 6 Gold tables (3 original + 3 enriched)
* 7 dashboard queries (4 original + 3 new for enriched data)
* Genie setup guide with semantic context
* MLflow experiment for supplier risk classification with enriched features
* AI function examples for contract intelligence
* Governance patterns (GRANT, RLS, tags)

### Next Steps
1. Run both notebooks in a live session with the Enbridge SCM team
2. Create the actual dashboards from the provided queries
3. Set up a Genie Space with the provided instructions
4. Configure access groups and permissions in production